# Create External Locations

Register ADLS Gen2 containers as Unity Catalog **External Locations** secured by a Storage Credential.

> **Prerequisites**
> - The Storage Credential `dbstoragefinancestoragetoken` must already exist in Unity Catalog (created via the Databricks Account Console or Terraform).
> - Run `0.config` first — all path and credential values come from Widgets so nothing is hardcoded here.

In [0]:
%run ./0.config

In [0]:
# ── Step 1: Register External Locations ──────────────────────────────────────

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {STORAGE_ACCOUNT}_bronze
URL '{BRONZE_PATH}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {STORAGE_ACCOUNT}_silver
URL '{SILVER_PATH}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {STORAGE_ACCOUNT}_gold
URL '{GOLD_PATH}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {STORAGE_ACCOUNT}_finance_{ENV}
URL '{CATALOG_ROOT}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
""")

In [0]:
# ── Step 2: Validate External Locations ──────────────────────────────────────

for loc in ["bronze", "silver", "gold", f"finance_{ENV}"]:
    loc_name = f"{STORAGE_ACCOUNT}_{loc}"
    df = spark.sql(f"DESC EXTERNAL LOCATION {loc_name}")
    print(f"\n── {loc_name} ──")
    df.display()

In [0]:
# ── Step 3: Smoke-test — read raw JSON directly from Bronze container ─────────
# Confirms the External Location and network routing are working end-to-end.

df = spark.read.option("multiLine", "true").json(f"{BRONZE_PATH}drivers.json")
print(f"drivers.json schema: {df.schema.simpleString()}")
df.display()